Dataset Source: IAM Handwriting Database(line level) - https://huggingface.co/datasets/Teklia/IAM-line

In [4]:
# Install Ollama and Python dependencies
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install jiwer -q
print("Setup Complete!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%##############################                  77.8%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "olla

In [5]:
# Start Ollama Server in Background
import os, time, subprocess

!pkill -9 ollama
time.sleep(2)

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

models = ["glm-ocr", "qwen2.5vl:3b", "llava:13b", "moondream"]
for m in models:
    print(f"Downloading {m}...")
    !ollama pull {m} > /dev/null 2>&1

print("All models ready!")

All models ready!


In [6]:
!pip install ollama datasets

In [7]:
import base64, re
from io import BytesIO
from datasets import load_dataset
from ollama import Client
import jiwer

client = Client(host='http://127.0.0.1:11434')

def evaluate_iam_words(model_name, num_samples=50):
    print(f"Starting IAM Handwriting Evaluation ({model_name}) on {num_samples} words...\n")
    
    print("Fetching dataset instantly via HuggingFace...")
    # This downloads the images instantly, bypassing Kaggle's UI completely
    dataset = load_dataset("Teklia/IAM-line", split="test")
    
    total_cer_acc = 0
    total_wer_acc = 0
    valid_samples = 0
    
    for idx, row in enumerate(dataset.select(range(num_samples)), 1):
        img = row['image']
        ground_truth_text = row['text']
        
        # Convert image to base64 for Ollama
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
        
        print(f"[{idx}/{num_samples}] Analyzing image (Ground Truth: '{ground_truth_text}')...")
        
        response = client.chat(
            model=model_name,
            messages=[{
                'role': 'user',
                'content': 'Transcribe the handwritten text in this image exactly as written. Output only the raw text, nothing else.',
                'images': [img_b64]
            }],
            options={
                'temperature': 0.0,
                'num_ctx': 4096,
                'num_predict': 50, 
                'repeat_penalty': 1.2,
                'stop': ['---', '```', '\n']
            }
        )
        
        predicted_text = response.get('message', {}).get('content', '').strip()
        
        clean_gt = re.sub(r'[^a-z0-9\s]', '', ground_truth_text.lower()).strip()
        clean_pred = re.sub(r'[^a-z0-9\s]', '', predicted_text.lower()).strip()
        
        if not clean_gt: clean_gt = "empty"
        if not clean_pred: clean_pred = "empty"
            
        cer = jiwer.cer(clean_gt, clean_pred)
        wer = jiwer.wer(clean_gt, clean_pred)
        
        cer_acc = max(0, (1 - cer) * 100)
        wer_acc = max(0, (1 - wer) * 100)
        
        print(f"Prediction: '{clean_pred}'")
        print(f"CER Accuracy: {cer_acc:.2f}% | WER Accuracy: {wer_acc:.2f}%")
        print("-" * 50)
        
        total_cer_acc += cer_acc
        total_wer_acc += wer_acc
        valid_samples += 1

    if valid_samples > 0:
        avg_cer_acc = total_cer_acc / valid_samples
        avg_wer_acc = total_wer_acc / valid_samples

        print("==================================================")
        print(f"MODEL                      : {model_name}")
        print(f"FINAL AVERAGE CER ACCURACY : {avg_cer_acc:.2f}%")
        print(f"FINAL AVERAGE WER ACCURACY : {avg_wer_acc:.2f}%")
        print("==================================================")

In [8]:
evaluate_iam_words('glm-ocr')

Starting IAM Handwriting Evaluation (glm-ocr) on 50 words...

Fetching dataset instantly via HuggingFace...


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/validation.parquet:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

data/test.parquet:   0%|          | 0.00/73.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6482 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/976 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2915 [00:00<?, ? examples/s]

[1/50] Analyzing image (Ground Truth: 'assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .')...
Prediction: 'assuredness  bella bella marie  parlophone  a lively song that changes tempo midway'
CER Accuracy: 97.65% | WER Accuracy: 100.00%
--------------------------------------------------
[2/50] Analyzing image (Ground Truth: 'I don't think he will storm the charts with this one , but it 's a good start .')...
Prediction: 'i dont think he will storm the charts with this one but it s a good start'
CER Accuracy: 98.65% | WER Accuracy: 100.00%
--------------------------------------------------
[3/50] Analyzing image (Ground Truth: 'CHRIS CHARLES , 39 , who lives in Stockton-on-Tees , is an accountant .')...
Prediction: 'chris charles 39 who lives in stocutonontees is an accountant'
CER Accuracy: 93.75% | WER Accuracy: 90.00%
--------------------------------------------------
[4/50] Analyzing image (Ground Truth: 'Become a success with a disc and h

In [9]:
evaluate_iam_words('qwen2.5vl:3b')

Starting IAM Handwriting Evaluation (qwen2.5vl:3b) on 50 words...

Fetching dataset instantly via HuggingFace...
[1/50] Analyzing image (Ground Truth: 'assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .')...
Prediction: 'assuredness bella bella marie parlophone a lively song that changes tempo midway'
CER Accuracy: 94.12% | WER Accuracy: 100.00%
--------------------------------------------------
[2/50] Analyzing image (Ground Truth: 'I don't think he will storm the charts with this one , but it 's a good start .')...
Prediction: 'i dont think he will storm the charts with this one but its a good start'
CER Accuracy: 97.30% | WER Accuracy: 88.24%
--------------------------------------------------
[3/50] Analyzing image (Ground Truth: 'CHRIS CHARLES , 39 , who lives in Stockton-on-Tees , is an accountant .')...
Prediction: 'chris charles 39 who lives in stocktonontees is an accountant'
CER Accuracy: 95.31% | WER Accuracy: 100.00%
---------------

In [10]:
evaluate_iam_words('llava:13b') 

Starting IAM Handwriting Evaluation (llava:13b) on 50 words...

Fetching dataset instantly via HuggingFace...
[1/50] Analyzing image (Ground Truth: 'assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .')...
Prediction: 'im sorry but there is no visible text or content to transcribe from the image you provided please ensure that the image contains legible text and try again if possible'
CER Accuracy: 0.00% | WER Accuracy: 0.00%
--------------------------------------------------
[2/50] Analyzing image (Ground Truth: 'I don't think he will storm the charts with this one , but it 's a good start .')...
Prediction: 'dont think he will close the chart with this one but it is good start do we have a good start'
CER Accuracy: 58.11% | WER Accuracy: 47.06%
--------------------------------------------------
[3/50] Analyzing image (Ground Truth: 'CHRIS CHARLES , 39 , who lives in Stockton-on-Tees , is an accountant .')...
Prediction: 'christopher charles 3

In [12]:
evaluate_iam_words('moondream') 

Starting IAM Handwriting Evaluation (moondream) on 50 words...

Fetching dataset instantly via HuggingFace...
[1/50] Analyzing image (Ground Truth: 'assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .')...
Prediction: 'empty'
CER Accuracy: 5.88% | WER Accuracy: 0.00%
--------------------------------------------------
[2/50] Analyzing image (Ground Truth: 'I don't think he will storm the charts with this one , but it 's a good start .')...
Prediction: 'empty'
CER Accuracy: 4.05% | WER Accuracy: 0.00%
--------------------------------------------------
[3/50] Analyzing image (Ground Truth: 'CHRIS CHARLES , 39 , who lives in Stockton-on-Tees , is an accountant .')...
Prediction: 'empty'
CER Accuracy: 3.12% | WER Accuracy: 0.00%
--------------------------------------------------
[4/50] Analyzing image (Ground Truth: 'Become a success with a disc and hey presto ! You 're a star ... . Rolly sings with')...
Prediction: 'empty'
CER Accuracy: 6.49% | WER